# 05 - Export pipeline: fit on `train_raw`, transform `train`, export final files

Thực hiện chạy pipeline chính thức từ `src/preprocessing.py` một cách độc lập và nhất quán, tạo bộ tham số và các file dữ liệu chuẩn hóa trong `data_processed/`.

**Quy tắc và ràng buộc:**
- Ràng buộc: Fit tiền xử lý **duy nhất trên `train_raw`**.
- Tập test luôn là tập thô (`test_raw.csv`), hoàn toàn độc lập và không chạm vào ở bước này.
- Tập validation không tách ở bước này; nếu cần dùng val ở bước evaluation chọn hyperparameters (bậc đa thức) thì sẽ tách trực tiếp từ `train`.

**Các bước thực hiện:**
1. Import `clean_raw`, `fit_preprocessing`, `transform` từ `src/preprocessing.py`.
2. Đọc `train_raw.csv` (80% dữ liệu từ `02_split.ipynb`), chạy `fit_preprocessing()` chỉ trên train.
3. Lưu `params` ra `data_processed/preprocessing_params.pkl`.
4. `transform()` cho `train_raw` tạo `X_train.csv` và `y_train.csv`.
5. **Sanity check cuối cùng**: Đọc lại file từ đĩa, kiểm tra không còn NaN, các cột số hợp lệ, mean ~ 0 và std ~ 1.

In [1]:
# Import libraries - ONLY use pandas/numpy/pickle, DO NOT use sklearn
import sys
import os
import pickle

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

def find_project_root(start=None, required_dirs=('src', 'data_processed'), max_levels=6):
    current = os.path.abspath(start or os.getcwd())
    for _ in range(max_levels):
        if all(os.path.isdir(os.path.join(current, name)) for name in required_dirs):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise RuntimeError(
        'Could not find the project root (needs both "src/" and "data_processed/"). '
        'Check whether you have run 02_split.ipynb to create data_processed/train_raw.csv yet.'
    )

print(f'Project root found: {PROJECT_ROOT}')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocessing import clean_raw, fit_preprocessing, transform

TRAIN_PATH = os.path.join(PROJECT_ROOT, 'data_processed', 'train_raw.csv')
TEST_PATH = os.path.join(PROJECT_ROOT, 'data_processed', 'test_raw.csv')
OUT_DIR = os.path.join(PROJECT_ROOT, 'data_processed')
PARAMS_PATH = os.path.join(OUT_DIR, 'preprocessing_params.pkl')

os.makedirs(OUT_DIR, exist_ok=True)
print('Finished importing from src/preprocessing.py')

Project root found: d:\ĐH\Năm 2\Học kỳ hè\Machine Learning\Source code compilation\Polynomial Regression
Finished importing from src/preprocessing.py


## 2. Fit `preprocessing` DUY NHẤT trên `train_raw`

In [2]:
train_raw = pd.read_csv(TRAIN_PATH, dtype=str, keep_default_na=False, na_values=[])
print(f'Shape train_raw: {train_raw.shape}')

train_clean = clean_raw(train_raw)
print(f'Shape train_clean (after clean_raw): {train_clean.shape}')

params = fit_preprocessing(train_clean)
print('\nKeys in params:', list(params.keys()))

Shape train_raw: (322, 9)
Shape train_clean (after clean_raw): (320, 9)

Keys in params: ['features', 'target', 'log_transform_cols', 'impute_values', 'outlier_bounds', 'scale_mean', 'scale_std']


## 3. Lưu `params` ra `data_processed/preprocessing_params.pkl`

In [3]:
with open(PARAMS_PATH, 'wb') as f:
    pickle.dump(params, f)

print(f'Saved params to: {PARAMS_PATH}')

Saved params to: d:\ĐH\Năm 2\Học kỳ hè\Machine Learning\Source code compilation\Polynomial Regression\data_processed\preprocessing_params.pkl


## 4. `transform()` cho `train_raw` và xuất file
Tập test vẫn giữ nguyên ở dạng thô (`test_raw.csv`). Nếu cần tập val để tinh chỉnh siêu tham số ở bước huấn luyện, val sẽ được tách từ `train`.

In [4]:
X_train, y_train = transform(train_raw, params)

print(f'X_train: {X_train.shape}   y_train: {y_train.shape}')

X_train_path = os.path.join(OUT_DIR, 'X_train.csv')
y_train_path = os.path.join(OUT_DIR, 'y_train.csv')

X_train.to_csv(X_train_path, index=False)
y_train.to_csv(y_train_path, index=False)

print('Saved successfully:')
print(f'  {X_train_path}  -> shape {X_train.shape}')
print(f'  {y_train_path}  -> shape {y_train.shape}')

X_train: (315, 7)   y_train: (315,)
Saved successfully:
  d:\ĐH\Năm 2\Học kỳ hè\Machine Learning\Source code compilation\Polynomial Regression\data_processed\X_train.csv  -> shape (315, 7)
  d:\ĐH\Năm 2\Học kỳ hè\Machine Learning\Source code compilation\Polynomial Regression\data_processed\y_train.csv  -> shape (315,)


## 5. Sanity check: Kiểm tra tính toàn vẹn dữ liệu
Đọc lại từ file trên đĩa để xác nhận:
- Không còn giá trị NaN
- 100% cột là số (`float64`)
- Mean ~ 0 và Std ~ 1 trên tập Train

In [5]:
X_train_check = pd.read_csv(X_train_path)
y_train_check = pd.read_csv(y_train_path)

print('=== 1) Check no remaining NaN ===')
nan_count_X = int(X_train_check.isna().sum().sum())
nan_count_y = int(y_train_check.isna().sum().sum())
print(f'NaN count in X_train.csv: {nan_count_X}')
print(f'NaN count in y_train.csv: {nan_count_y}')
assert nan_count_X == 0, 'ERROR: X_train still has NaN!'
assert nan_count_y == 0, 'ERROR: y_train still has NaN!'
print('=> Passed: no remaining NaN.\n')

print('=== 2) Check no string values in numeric columns ===')
non_numeric_cols = [c for c in X_train_check.columns if not pd.api.types.is_numeric_dtype(X_train_check[c])]
print(f'Columns that are NOT numeric (should be empty): {non_numeric_cols}')
assert len(non_numeric_cols) == 0, f'ERROR: found non-numeric columns: {non_numeric_cols}'
assert pd.api.types.is_numeric_dtype(y_train_check[y_train_check.columns[0]]), 'ERROR: y_train is not numeric!'
print('=> Passed: all columns are numeric.\n')

print('=== 3) Mean / Std of each standardized column (expected ~ 0 and ~ 1) ===')
mean_std_table = X_train_check.describe().T[['mean', 'std']].round(4)
print(mean_std_table)

max_mean_diff = mean_std_table['mean'].abs().max()
max_std_diff = (mean_std_table['std'] - 1).abs().max()
print(f'\nLargest mean deviation from 0: {max_mean_diff:.4f}')
print(f'Largest std deviation from 1  : {max_std_diff:.4f}')
assert max_mean_diff < 0.05, 'ERROR: a column mean deviates too far from 0!'
assert max_std_diff < 0.05, 'ERROR: a column std deviates too far from 1!'
print('=> Passed: standardization is as expected.')

print('\n===== SANITY CHECK: ALL CONDITIONS PASSED =====')

=== 1) Check no remaining NaN ===
NaN count in X_train.csv: 0
NaN count in y_train.csv: 0
=> Passed: no remaining NaN.

=== 2) Check no string values in numeric columns ===
Columns that are NOT numeric (should be empty): []
=> Passed: all columns are numeric.

=== 3) Mean / Std of each standardized column (expected ~ 0 and ~ 1) ===
                mean     std
cylinders     0.0063  1.0057
displacement  0.0065  1.0058
horsepower    0.0013  1.0094
weight        0.0059  1.0063
acceleration  0.0038  1.0083
model year    0.0072  0.9978
origin       -0.0082  0.9977

Largest mean deviation from 0: 0.0082
Largest std deviation from 1  : 0.0094
=> Passed: standardization is as expected.

===== SANITY CHECK: ALL CONDITIONS PASSED =====


## Kết luận
- Pipeline tiền xử lý được thực hiện hoàn toàn độc lập và chỉ fit tham số trên `train_raw`.
- Bộ tham số được lưu thành file `preprocessing_params.pkl` có thể tái sử dụng cho bất kỳ dữ liệu mới nào.
- `test_raw.csv` hoàn toàn độc lập và được giữ ở trạng thái thô để đánh giá cuối cùng.